# Nepali TTS — Colab training shift (checkpoint relay)

This notebook lets a **free Colab GPU** take a training shift on the *same* run as the home laptop.
The full-state checkpoint (the "baton") is passed through a private Hugging Face repo, and a lock
stops the two machines from ever training at once. **No progress is lost** — Colab pulls the latest
checkpoint, trains, and pushes it back (every ~20 min and on exit).

**Before running:** Runtime ▸ Change runtime type ▸ **GPU**. Then add your HF token as a Colab secret
(the 🔑 icon on the left) named `HF_TOKEN`. Then Runtime ▸ **Run all**.

> First-time note: the environment install (cell 3) takes a few minutes and may need a tweak for
> Colab's current CUDA/torch — run it once and check the import line at the end says OK.

In [ ]:
# 1) Check we actually got a GPU
!nvidia-smi -L
import torch
print('torch', torch.__version__, '| cuda', torch.version.cuda, '| gpu', torch.cuda.is_available())

In [ ]:
# 2) Credentials + repo names
import os
try:
    from google.colab import userdata
    os.environ['HF_TOKEN'] = userdata.get('HF_TOKEN')
    print('token loaded from Colab secret')
except Exception:
    import getpass
    os.environ['HF_TOKEN'] = getpass.getpass('Paste your HF write token: ')
os.environ['HF_REPO']   = 'byapaksigdel/nepali-tts-ckpt'
os.environ['DATA_REPO'] = 'byapaksigdel/nepali-tts-data'
os.environ['DEVICE_ID'] = 'colab'
print('repo:', os.environ['HF_REPO'])

In [ ]:
# 3) Install: code repo, espeak-ng, huggingface_hub, and the piper1-gpl trainer
!git clone -q https://github.com/ByapakSigdel/nepali-tts.git /content/nepali-tts || (cd /content/nepali-tts && git pull -q)
!apt-get -qq install -y espeak-ng >/dev/null && echo 'espeak-ng OK'
!pip -q install 'huggingface_hub>=0.24' soundfile ninja
print('torch BEFORE:', __import__('torch').__version__)
!git clone -q https://github.com/OHF-Voice/piper1-gpl.git /content/piper1-gpl
%cd /content/piper1-gpl
!pip -q install -e '.[train]'
!bash ./build_monotonic_align.sh
!python setup.py build_ext --inplace >/dev/null 2>&1 && echo 'build_ext OK'
import torch; print('torch AFTER:', torch.__version__, '| gpu', torch.cuda.is_available())
!python -c 'import piper.train; print("piper.train import OK")'
%cd /content/nepali-tts

In [ ]:
# 4) Download the training audio (one-time per session) from the private dataset repo
import os, tarfile
from huggingface_hub import hf_hub_download
p = hf_hub_download(os.environ['DATA_REPO'], 'processed.tar.gz', repo_type='dataset', token=os.environ['HF_TOKEN'])
os.makedirs('/content/data', exist_ok=True)
with tarfile.open(p) as t:
    t.extractall('/content/data')
wavs = os.listdir('/content/data/processed/wavs')
print('dataset ready:', len(wavs), 'clips')

In [ ]:
# 5) Take the lock + pull the baton (config + latest checkpoint)
import os, sys, subprocess, shutil
SCRIPTS = '/content/nepali-tts/scripts'
os.makedirs('/content/run/ne_stageA/ckpts', exist_ok=True)
def hs(*a):
    return subprocess.run([sys.executable, f'{SCRIPTS}/hf_sync.py', *a], env=os.environ).returncode
rc = hs('claim')
assert rc == 0, 'Another machine holds the training lock. Stop the laptop first, or wait ~30 min for it to auto-free.'
from huggingface_hub import hf_hub_download
cfg = hf_hub_download(os.environ['HF_REPO'], 'config.json', token=os.environ['HF_TOKEN'])
shutil.copy(cfg, '/content/run/ne_stageA/config.json')
hs('pull', 'last.ckpt', '/content/run/ne_stageA/ckpts/last.ckpt')
print('remote epoch ->', end=' '); hs('remote-epoch')

In [ ]:
# 6) TRAIN. A background thread pushes the checkpoint every ~20 min (Colab can drop suddenly).
#    On exit (finish / disconnect / Ctrl-C) it does a final push + releases the lock.
import os, sys, re, time, threading, subprocess
SCRIPTS = '/content/nepali-tts/scripts'
RUN = '/content/run/ne_stageA'
CKPT = f'{RUN}/ckpts/last.ckpt'; STATUS = f'{RUN}/status.txt'
os.environ['PYTHONPATH'] = SCRIPTS + ':' + os.environ.get('PYTHONPATH', '')
TARGET = 350

def hs(*a):
    return subprocess.run([sys.executable, f'{SCRIPTS}/hf_sync.py', *a], env=os.environ).returncode
def epoch_now():
    try:
        m = re.search(r'(?:^| )epoch=(\d+)', open(STATUS).read())
        return int(m.group(1)) if m else -1
    except Exception:
        return -1

_stop = False
def pusher():
    while not _stop:
        t0 = time.time()
        while time.time() - t0 < 20 * 60:
            if _stop:
                return
            time.sleep(2)
        hs('heartbeat')
        if os.path.exists(CKPT):
            hs('push', CKPT, 'last.ckpt'); hs('push-progress', str(epoch_now()))
threading.Thread(target=pusher, daemon=True).start()

cmd = ['python', '-m', 'piper.train', 'fit',
       '--config', '/content/nepali-tts/configs/train_ne_colab.yaml',
       '--data.voice_name', 'ne_stageA',
       '--data.csv_path', '/content/data/processed/metadata.csv',
       '--data.audio_dir', '/content/data/processed/wavs',
       '--data.espeak_voice', 'ne',
       '--data.cache_dir', f'{RUN}/cache',
       '--data.config_path', f'{RUN}/config.json',
       '--data.batch_size', '8',
       '--data.num_workers', '2',
       '--model.sample_rate', '22050',
       '--model.num_speakers', '20',
       '--trainer.accelerator', 'gpu', '--trainer.devices', '1',
       '--trainer.precision', '32-true',
       '--trainer.max_epochs', str(TARGET),
       '--trainer.default_root_dir', RUN,
       '--trainer.num_sanity_val_steps', '0',
       '--trainer.limit_val_batches', '0',
       '--trainer.log_every_n_steps', '25',
       '--ckpt_path', CKPT]

print('Training on Colab GPU -> epoch', TARGET, '. Keep this tab open.')
try:
    for attempt in range(1, 100):
        if epoch_now() >= TARGET:
            print('reached target'); break
        print(f'--- attempt {attempt} ---')
        code = subprocess.run(cmd, env=os.environ).returncode
        if code == 0:
            print('training finished cleanly'); break
        print(f'exited {code}; resuming in 8s'); time.sleep(8)
finally:
    _stop = True
    if os.path.exists(CKPT):
        hs('push', CKPT, 'last.ckpt'); hs('push-progress', str(epoch_now()))
    hs('release')
    print('FINAL push done + lock released. Safe to close. epoch =', epoch_now())

## When you're done (or Colab kicks you off)

The last cell already pushed the checkpoint and released the lock, so the home laptop can pull the
progress and continue — just start training on the laptop again. If Colab disconnects abruptly,
at most the last ~20 minutes (since the previous auto-push) is re-done; nothing is lost.

To see who currently holds the lock from anywhere:
`!python /content/nepali-tts/scripts/hf_sync.py status`